In [44]:
import polars as pl
import scipy.sparse as sp
import numpy as np
import joblib
import os

# --- 1. LOAD THE CLEANED DATA ONLY ---
# We use the 'final' version we just created!
data_path = "data/processed/ratings_final.parquet" 

print(f"--- Loading Filtered Data from: {data_path} ---")
df = pl.read_parquet(data_path)

# --- 2. Create Mappings ---
# We need to map real IDs (like 163668) to matrix indices (0, 1, 2...)
print("--- Creating Mappings ---")
user_ids = df["userId"].unique().sort()
movie_ids = df["movieId"].unique().sort()

user_to_index = {user: i for i, user in enumerate(user_ids)}
movie_to_index = {movie: i for i, movie in enumerate(movie_ids)}

# SAVE THESE - You will need them for the prediction app!
joblib.dump(user_to_index, "user_map.pkl")
joblib.dump(movie_to_index, "movie_map.pkl")
print(f"Mappings saved. Users: {len(user_ids)}, Movies: {len(movie_ids)}")

# --- 3. Build the Interaction Matrix ---
print("--- Building Interaction Matrix (This may take a minute) ---")

# Replace real IDs with our new 0-indexed positions
df = df.with_columns([
    pl.col("userId").replace(user_to_index).alias("user_idx"),
    pl.col("movieId").replace(movie_to_index).alias("movie_idx")
])

# Convert to numpy for Scipy
rows = df["user_idx"].to_numpy()
cols = df["movie_idx"].to_numpy()
data = df["rating"].to_numpy().astype(np.float32)

# Create the sparse matrix (COO format is fastest for building)
interaction_matrix = sp.coo_matrix(
    (data, (rows, cols)), 
    shape=(len(user_ids), len(movie_ids))
)

# Convert to CSR format (Faster for the AI to actually read)
interaction_matrix = interaction_matrix.tocsr()

print("--- SUCCESS ---")
print(f"Matrix Shape: {interaction_matrix.shape} (Users x Movies)")

--- Loading Filtered Data from: data/processed/ratings_final.parquet ---
--- Creating Mappings ---
Mappings saved. Users: 200948, Movies: 43884
--- Building Interaction Matrix (This may take a minute) ---
--- SUCCESS ---
Matrix Shape: (200948, 43884) (Users x Movies)


In [ ]:
import polars as pl
import os

# 1. Use the FINAL processed paths
processed_ratings = "data/processed/ratings_final.parquet"
processed_movies = "data/processed/movies_final.parquet"

# 2. Load the cleaned files (Fast as hell)
df_ratings = pl.read_parquet(processed_ratings)
df_movies = pl.read_parquet(processed_movies)

# 3. Peek at the data
print(f"--- RATINGS: {df_ratings.height} ROWS ---")
print(df_ratings.head(5))

print(f"\n--- MOVIES: {df_movies.height} ROWS ---")
print(df_movies.head(5))

# 4. Check Schema
print("\n--- DATA TYPES ---")
print("Ratings Schema:", df_ratings.schema)
print("Movies Schema:", df_movies.schema)

--- RATINGS: 31921467 ROWS ---
shape: (5, 3)
┌────────┬─────────┬────────┐
│ userId ┆ movieId ┆ rating │
│ ---    ┆ ---     ┆ ---    │
│ i64    ┆ i64     ┆ f64    │
╞════════╪═════════╪════════╡
│ 1      ┆ 17      ┆ 4.0    │
│ 1      ┆ 25      ┆ 1.0    │
│ 1      ┆ 29      ┆ 2.0    │
│ 1      ┆ 30      ┆ 5.0    │
│ 1      ┆ 32      ┆ 5.0    │
└────────┴─────────┴────────┘

--- MOVIES: 43884 ROWS ---
shape: (5, 5)
┌─────────┬───────────────────────────┬──────────────────────────┬──────┬──────────────────────────┐
│ movieId ┆ title                     ┆ genres                   ┆ year ┆ title_clean              │
│ ---     ┆ ---                       ┆ ---                      ┆ ---  ┆ ---                      │
│ i64     ┆ str                       ┆ str                      ┆ i32  ┆ str                      │
╞═════════╪═══════════════════════════╪══════════════════════════╪══════╪══════════════════════════╡
│ 1       ┆ Toy Story (1995)          ┆ Adventure|Animation|Chil ┆ 1995 ┆ Toy 

: 

In [ ]:
# ONLY keep this if you want to see the status of the FINAL data
import polars as pl

# Load the CLEANED files we just saved
movies_df = pl.read_parquet("data/processed/movies_final.parquet")
ratings_df = pl.read_parquet("data/processed/ratings_final.parquet")

print(f"--- STATUS CHECK ---")
print(f"Movies: {movies_df.height} (Should be 43,884)")
print(f"Ratings: {ratings_df.height} (Should be 31,921,467)")
print(f"Nulls in Movies:\n{movies_df.null_count()}")

--- STATUS CHECK ---
Movies: 43884 (Should be 43,884)
Ratings: 31921467 (Should be 31,921,467)
Nulls in Movies:
shape: (1, 5)
┌─────────┬───────┬────────┬──────┬─────────────┐
│ movieId ┆ title ┆ genres ┆ year ┆ title_clean │
│ ---     ┆ ---   ┆ ---    ┆ ---  ┆ ---         │
│ u32     ┆ u32   ┆ u32    ┆ u32  ┆ u32         │
╞═════════╪═══════╪════════╪══════╪═════════════╡
│ 0       ┆ 0     ┆ 0      ┆ 0    ┆ 0           │
└─────────┴───────┴────────┴──────┴─────────────┘


: 

In [ ]:
import polars as pl
import os

# 1. RE-LOAD THE RAW TAGS (Since they got dropped from memory)
# We need to grab them from the source one last time to merge them
tags_path = os.path.join("data", "raw", "tags.csv")
tags_df = (
    pl.read_csv(tags_path)
    .select(["movieId", "tag"])
    .drop_nulls()
    .with_columns(pl.col("tag").str.to_lowercase())
    .group_by("movieId")
    .agg(pl.col("tag").str.join(" "))
)

# 2. MERGE THEM INTO YOUR MOVIE LIST
# We use a left join: if a movie has no tags, it stays in the list
movies_df = movies_df.join(tags_df, on="movieId", how="left")

# 3. NOW FILL THE NULLS (This is where the previous code failed)
movies_df = movies_df.with_columns(pl.col("tag").fill_null(""))

print("--- REPAIR SUCCESSFUL ---")
print(f"Movies with tags: {movies_df.filter(pl.col('tag') != '').height} out of {movies_df.height}")
print(f"Columns now: {movies_df.columns}")

--- REPAIR SUCCESSFUL ---
Movies with tags: 35231 out of 43884
Columns now: ['movieId', 'title', 'genres', 'year', 'title_clean', 'tag']


: 

In [ ]:
import polars as pl
import os

# Load the final files we just saved
PATH_RATINGS_FINAL = os.path.join("data", "processed", "ratings_final.parquet")
ratings_df = pl.read_parquet(PATH_RATINGS_FINAL)

# 1. Check User Activity (How many ratings per user?)
user_counts = ratings_df.group_by("userId").len().sort("len")

# 2. Check Movie Popularity (How many ratings per movie?)
movie_counts = ratings_df.group_by("movieId").len().sort("len")

print("--- USER ACTIVITY STATS ---")
print(f"Total Users: {len(user_counts)}")
print(f"Users with only 1 rating: {user_counts.filter(pl.col('len') == 1).height}")
print(f"Average ratings per user: {user_counts['len'].mean():.2f}")

print("\n--- MOVIE POPULARITY STATS ---")
print(f"Total Movies: {len(movie_counts)}")
print(f"Movies with only 1 rating: {movie_counts.filter(pl.col('len') == 1).height}")

--- USER ACTIVITY STATS ---
Total Users: 200948
Users with only 1 rating: 0
Average ratings per user: 158.85

--- MOVIE POPULARITY STATS ---
Total Movies: 43884
Movies with only 1 rating: 0


: 

In [ ]:
# Create a list of movieIds that have at least 5 ratings
movie_counts = ratings_df.group_by("movieId").len()
keep_ids = movie_counts.filter(pl.col("len") >= 5).select("movieId")

print(f"Number of movies that have 5+ ratings: {keep_ids.height}")

Number of movies that have 5+ ratings: 43884


: 

In [ ]:
# STEP 3: Filter the movies and drop the null title
# We use an inner join to keep ONLY the movies in our 'keep_ids' list
movies_df = movies_df.join(keep_ids, on="movieId", how="inner")

# Drop the 1 row where title_clean is missing
movies_df = movies_df.drop_nulls(subset=["title_clean"])

print(f"Final Movie Count: {movies_df.height}")

Final Movie Count: 43884


: 

In [ ]:
# 1. Filter the ratings to match the 43,884 movies we kept
ratings_df = ratings_df.join(keep_ids, on="movieId", how="inner")

# 2. Save both to the disk (Overwriting the old messy files)
movies_df.write_parquet("data/processed/movies_final.parquet")
ratings_df.write_parquet("data/processed/ratings_final.parquet")

print("--- FILES UPDATED ON DISK ---")
print(f"Movies saved: {movies_df.height} rows")
print(f"Ratings saved: {ratings_df.height} rows")

--- FILES UPDATED ON DISK ---
Movies saved: 43884 rows
Ratings saved: 31921467 rows


: 

In [1]:
import polars as pl
import os

# 1. LOAD DATA (Reading only what we need to save RAM)
print("Loading data...")
movies_df = pl.read_parquet("data/processed/movies_final.parquet")
ratings_df = pl.read_parquet("data/processed/ratings_final.parquet")
tags_path = os.path.join("data", "raw", "tags.csv")

# 2. PROCESS TAGS
# We process tags and join them immediately to movies_df
tags_df = (
    pl.scan_csv(tags_path)
    .select(["movieId", "tag"])
    .drop_nulls()
    .with_columns(pl.col("tag").str.to_lowercase())
    .group_by("movieId")
    .agg(pl.col("tag").str.join(" "))
    .collect()
)

movies_df = movies_df.join(tags_df, on="movieId", how="left").with_columns(
    pl.col("tag").fill_null("")
)

# 3. DEFINE THE KEEP_IDS (Sanity Check)
movie_counts = ratings_df.group_by("movieId").len()
keep_ids = movie_counts.filter(pl.col("len") >= 5).select("movieId")

# 4. FINAL FILTER & SAVE
# This ensures both files are perfectly synced at 43,884 movies
movies_df = movies_df.join(keep_ids, on="movieId", how="inner")
ratings_df = ratings_df.join(keep_ids, on="movieId", how="inner")

# Overwrite the clean files with the NEW column included
movies_df.write_parquet("data/processed/movies_final.parquet")
ratings_df.write_parquet("data/processed/ratings_final.parquet")

print("\n--- SYSTEM RECOVERED & FILES UPDATED ---")
print(f"Movies: {movies_df.height} | Ratings: {ratings_df.height}")
print(f"Columns: {movies_df.columns}")

Loading data...

--- SYSTEM RECOVERED & FILES UPDATED ---
Movies: 43884 | Ratings: 31921467
Columns: ['movieId', 'title', 'genres', 'year', 'title_clean', 'tag', 'tag_right']


In [ ]:
# STEP 4 & 5: Feature Engineering (The AI Prep)

# 1. Clean up redundant columns from the join
if "tag_right" in movies_df.columns:
    movies_df = movies_df.drop("tag_right")

# 2. Get Unique Genres
unique_genres = (
    movies_df.select("genres")
    .with_columns(pl.col("genres").str.split("|"))
    .explode("genres")
    .unique()
    .sort("genres")
)["genres"].to_list()

if "(no genres listed)" in unique_genres: unique_genres.remove("(no genres listed)")

# 3. Multi-Hot Encode Genres (Create the g_ columns)
for genre in unique_genres:
    col_name = f"g_{genre.lower().replace(' ', '_').replace('-', '_')}"
    movies_df = movies_df.with_columns(
        pl.col("genres").str.contains(genre).cast(pl.Int8).alias(col_name)
    )

# 4. Scale the Year (Min-Max Scaling)
min_y = movies_df.filter(pl.col("year") > 0)["year"].min()
max_y = movies_df["year"].max()
movies_df = movies_df.with_columns(
    ((pl.col("year") - min_y) / (max_y - min_y)).clip(0, 1).alias("year_scaled")
)

# 5. Save this 'Featurized' version
movies_df.write_parquet("data/processed/movies_featurized.parquet")

print("--- FEATURE ENGINEERING COMPLETE ---")
print(f"New Columns: {[c for c in movies_df.columns if c.startswith('g_') or c == 'year_scaled']}")
print(f"Final Table saved to: data/processed/movies_featurized.parquet")

In [2]:
# Kill the redundant tag_right column
if "tag_right" in movies_df.columns:
    movies_df = movies_df.drop("tag_right")

# Verify it's gone
print("Remaining Columns:")
print(movies_df.columns)

Remaining Columns:
['movieId', 'title', 'genres', 'year', 'title_clean', 'tag']


In [3]:
# 1. Drop the original messy 'title'
movies_df = movies_df.drop("title")

# 2. Rename 'title_clean' to 'title' so it looks normal
movies_df = movies_df.rename({"title_clean": "title"})

# 3. Save the clean version back to disk
movies_df.write_parquet("data/processed/movies_final.parquet")

print("--- REDUNDANCY REMOVED ---")
print(f"Current Columns: {movies_df.columns}")
print(movies_df.select(["movieId", "title", "year"]).head(5))

--- REDUNDANCY REMOVED ---
Current Columns: ['movieId', 'genres', 'year', 'title', 'tag']
shape: (5, 3)
┌─────────┬────────────────────────────┬──────┐
│ movieId ┆ title                      ┆ year │
│ ---     ┆ ---                        ┆ ---  │
│ i64     ┆ str                        ┆ i32  │
╞═════════╪════════════════════════════╪══════╡
│ 285757  ┆ Supercell                  ┆ 2023 │
│ 133743  ┆ Kokoko                     ┆ 2012 │
│ 216845  ┆ Malang                     ┆ 2020 │
│ 177923  ┆ Evil Dead Trap             ┆ 1988 │
│ 74480   ┆ Other End of the Line, The ┆ 2008 │
└─────────┴────────────────────────────┴──────┘


In [4]:
# 1. See all unique titles as a list (first 20)
print("--- FIRST 20 UNIQUE TITLES ---")
print(movies_df.select("title").unique().head(20))

# 2. THE HELPFUL SHIT: See if any titles are duplicates
# (This shows you titles that appear in the list more than once)
duplicates = (
    movies_df.group_by("title")
    .len()
    .filter(pl.col("len") > 1)
    .sort("len", descending=True)
)

print("\n--- DUPLICATED TITLES (Movies with the same name) ---")
print(duplicates.head(10))

--- FIRST 20 UNIQUE TITLES ---
shape: (20, 1)
┌──────────────────────────┐
│ title                    │
│ ---                      │
│ str                      │
╞══════════════════════════╡
│ Lords of Discipline, The │
│ Gigi & Nate              │
│ The Amputee              │
│ Fire on the Amazon       │
│ Barbie: Spy Squad        │
│ …                        │
│ Paul Blart: Mall Cop 2   │
│ Like a Boss              │
│ Lay of the Land, The     │
│ Roots                    │
│ Cyborg Cop II            │
└──────────────────────────┘

--- DUPLICATED TITLES (Movies with the same name) ---
shape: (10, 2)
┌───────────────────┬─────┐
│ title             ┆ len │
│ ---               ┆ --- │
│ str               ┆ u32 │
╞═══════════════════╪═════╡
│ Cinderella        ┆ 12  │
│ Hamlet            ┆ 8   │
│ Misérables, Les   ┆ 7   │
│ Treasure Island   ┆ 7   │
│ Home              ┆ 6   │
│ Paradise          ┆ 6   │
│ Holiday           ┆ 6   │
│ Pinocchio         ┆ 6   │
│ Wuthering Heights ┆ 6   │

In [5]:
# 1. Create a unique display title by combining title and year
# This turns "Cinderella" into "Cinderella (1950)"
movies_df = movies_df.with_columns(
    (pl.col("title") + " (" + pl.col("year").cast(pl.Utf8) + ")").alias("display_title")
)

# 2. Check for duplicates again on this NEW column
final_dupes = (
    movies_df.group_by("display_title")
    .len()
    .filter(pl.col("len") > 1)
)

print(f"Duplicate 'display_titles' remaining: {final_dupes.height}")
print("--- TITLES ARE NOW UNIQUE ---")
print(movies_df.select(["display_title", "genres"]).head(5))

Duplicate 'display_titles' remaining: 69
--- TITLES ARE NOW UNIQUE ---
shape: (5, 2)
┌─────────────────────────────────┬──────────────────────┐
│ display_title                   ┆ genres               │
│ ---                             ┆ ---                  │
│ str                             ┆ str                  │
╞═════════════════════════════════╪══════════════════════╡
│ Supercell (2023)                ┆ Action               │
│ Kokoko (2012)                   ┆ (no genres listed)   │
│ Malang (2020)                   ┆ Action|Drama|Romance │
│ Evil Dead Trap (1988)           ┆ Horror               │
│ Other End of the Line, The (20… ┆ Comedy|Romance       │
└─────────────────────────────────┴──────────────────────┘


In [6]:
# THE FINAL LOCK-IN
# 1. Update the saved files with the 'display_title' and cleaned columns
movies_df.write_parquet("data/processed/movies_final.parquet")
ratings_df.write_parquet("data/processed/ratings_final.parquet")

print("--- FILES SAVED TO DISK FOREVER ---")
print(f"Final Movies: {movies_df.height} (with unique display_titles)")
print(f"Final Ratings: {ratings_df.height}")
print(f"Columns in movies_final: {movies_df.columns}")

--- FILES SAVED TO DISK FOREVER ---
Final Movies: 43884 (with unique display_titles)
Final Ratings: 31921467
Columns in movies_final: ['movieId', 'genres', 'year', 'title', 'tag', 'display_title']
